Полное SFT: Qwen3.5-2B, LoRA, Spider train 5819 примеров, 1 эпоха.

В Kaggle включите GPU T4, Internet, добавьте прежний датасет и Secret COMET_API_KEY с доступом этому notebook. Выполняйте ячейки последовательно. Подбор гиперпараметров и Base-оценка не повторяются. Используется одна GPU.

Конфигурация: LR 1e-4, микробатч 1, накопление 8, LoRA r16/alpha32, warmup 3%, linear decay. Чекпоинты каждые 50 шагов загружаются в Comet; сообщение Comet confirmed backup подтверждает доставку. При сбросе можно потерять работу после последнего доставленного чекпоинта.

Первая ячейка ставит зависимости. Исправление cuDNN выполняется последним: повторный общий pip install может вернуть прежнюю cuDNN. Если до установки уже импортировали torch, после установки перезапустите kernel и продолжайте со второй ячейки.

In [ ]:
%pip install "vllm==0.31.0" "transformers==5.17.0" "peft==0.21.2" "accelerate==1.15.0" "comet-ml==3.59.0" python-dotenv
%pip install --no-deps --index-url https://download.pytorch.org/whl/cu130 "torchaudio==2.11.0+cu130"
%pip uninstall -y torchao torchcodec
%pip install --no-deps --no-cache-dir --index-url https://download.pytorch.org/whl "nvidia-cudnn-cu13==9.23.0.39"

Подготовка репозитория и данных. DATA_DIR можно указать вручную, если автоматический поиск не нашёл прежнюю структуру Spider и BIRD.

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime, timezone

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "text2sql-grpo-sft"
RUN = WORK / "text2sql_runs" / "v1"
BASE = "Qwen/Qwen3.5-2B"
REVISION = "15852e8c16360a2fea060d615a32b45270f8a8fc"

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/dancravik/text2sql-grpo.git", str(REPO)], check=True)

DATA_DIR = None  # При необходимости укажите Path("/kaggle/input/.../...")
if DATA_DIR is None:
    roots = sorted({
        p.parents[1] for p in INPUT.rglob("train_spider.json")
        if p.parent.name == "spider"
        and (p.parents[1] / "minidev/MINIDEV/mini_dev_sqlite.json").is_file()
    })
    if len(roots) == 1:
        DATA_DIR = roots[0]
    elif roots:
        raise ValueError(f"Укажите DATA_DIR вручную: {roots}")
    else:
        DATA_DIR = WORK / "text2sql_data"
        if not (DATA_DIR / "spider/train_spider.json").is_file():
            archives = list(INPUT.rglob("text2sql-data.zip"))
            if len(archives) != 1:
                raise ValueError(f"Добавьте прежний датасет или задайте DATA_DIR: {archives}")
            with ZipFile(archives[0]) as archive:
                archive.extractall(DATA_DIR)

RUN.mkdir(parents=True, exist_ok=True)
os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.pop("TORCH_CUDNN_V8_API_DISABLED", None)
subprocess.run([sys.executable, "scripts/freeze_splits.py", "--verify"], cwd=REPO, check=True)
print("DATA_DIR:", DATA_DIR)
print("Свободно GiB:", round(shutil.disk_usage(WORK).free / 1024**3, 2))

Ключ Comet и проверка GPU. Ключ не выводится. Workspace указан для текущего проекта; при необходимости измените его.

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["COMET_API_KEY"] = UserSecretsClient().get_secret("COMET_API_KEY")
os.environ["COMET_WORKSPACE"] = "danchenko-rtm"
os.environ["COMET_PROJECT_NAME"] = "text2sql-grpo"

subprocess.run([sys.executable, "-u", "-c", r"""
import importlib.metadata
import torch
from transformers import Trainer
from peft import LoraConfig
from vllm import LLM
for package in ("torch", "transformers", "peft", "accelerate", "vllm", "comet-ml", "nvidia-cudnn-cu13"):
    print(package, importlib.metadata.version(package))
assert torch.cuda.device_count() == 1
x = torch.randn(1, 6144, 512, device="cuda", dtype=torch.float16, requires_grad=True)
w = torch.randn(6144, 1, 4, device="cuda", dtype=torch.float16)
torch.nn.functional.conv1d(x, w, groups=6144, padding=3).float().mean().backward()
torch.cuda.synchronize()
print("GPU и FP16 conv1d forward/backward OK")
from comet_tracking import start_experiment
experiment = start_experiment(name="sft_full_environment_check")
try:
    experiment.log_metric("environment_ready", 1)
finally:
    experiment.end()
"""], cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO / "src")}, check=True)

Полное обучение. Создаётся новый запуск с префиксом sft_full_. Не задаём num-train-examples: используется весь train. Если нужно продолжить после сброса, распакуйте checkpoint-N.zip из Comet и укажите путь к checkpoint-N в RESUME_CHECKPOINT. Настройки и исходные данные должны совпадать; LR-расписание продолжится с сохранённого шага. При новом запуске и при продолжении создаётся новый локальный каталог. При продолжении сохраняется исходный эксперимент Comet.

In [ ]:
RESUME_CHECKPOINT = None  # Например Path("/kaggle/working/restored/checkpoint-100")

name = "sft_full_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
FULL = RUN / name
if FULL.exists():
    raise FileExistsError(FULL)
if shutil.disk_usage(WORK).free < 10 * 1024**3:
    raise RuntimeError("Оставьте минимум 10 GiB диска перед загрузкой модели и обучением.")

logs = RUN / "logs"
logs.mkdir(exist_ok=True)
TRAIN_LOG = logs / f"{name}.log"
command = [
    sys.executable, "-u", str(REPO / "scripts/train_sft.py"),
    "--model", BASE, "--revision", REVISION,
    "--manifest", str(REPO / "splits/v1.json"),
    "--out-dir", str(FULL),
    "--epochs", "1", "--seed", "42",
    "--learning-rate", "1e-4",
    "--micro-batch-size", "1", "--grad-accum", "8",
    "--lora-r", "16", "--lora-alpha", "32",
    "--max-length", "4096", "--warmup-ratio", "0.03",
    "--logging-steps", "5",
    "--save-strategy", "steps", "--save-steps", "50",
    "--comet", "--comet-checkpoints",
]
if RESUME_CHECKPOINT is not None:
    command += ["--resume-from-checkpoint", str(RESUME_CHECKPOINT)]

print("Полный SFT:", FULL, flush=True)
with TRAIN_LOG.open("w", encoding="utf-8") as logfile:
    with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            logfile.write(line)
            logfile.flush()
        exit_code = process.wait()

if exit_code != 0:
    raise RuntimeError(f"Процесс завершился с кодом {exit_code}. Лог: {TRAIN_LOG}. Не запускайте обучение повторно без проверки чекпоинтов.")

report = json.loads((FULL / "run.json").read_text())
assert report["num_train_examples"] == 5819, report["num_train_examples"]
print("Примеров:", report["num_train_examples"])
print("Применено обновлений:", report["optimizer_steps_applied"])
print("Пропущено обновлений:", report["optimizer_steps_skipped"])
print("Отчёт:", FULL / "run.json")

После успешного обучения: оценка на полной validation, без тестовых выборок. Эта ячейка не изменяет веса. Инициализация vLLM на T4 может занять несколько минут.

In [ ]:
EVAL_NAME = FULL.name
EVAL = RUN / "eval" / "validation" / EVAL_NAME
EVAL_LOG = RUN / "logs" / f"{EVAL_NAME}_validation.log"
command = [
    sys.executable, "-u", str(REPO / "scripts/evaluate.py"),
    "--model", str(FULL / "merged"), "--name", EVAL_NAME,
    "--datasets", "validation", "--out-dir", str(EVAL.parent),
    "--comet-run", str(FULL / "run.json"),
    "--n", "1", "--temperature", "0", "--enable-thinking", "off",
    "--max-tokens", "1024", "--max-model-len", "4096", "--seed", "42",
]
if not (EVAL / "metrics.json").exists():
    if EVAL.exists():
        raise RuntimeError(f"Каталог незавершённой оценки уже существует: {EVAL}")
    with EVAL_LOG.open("w", encoding="utf-8") as logfile:
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                logfile.write(line)
                logfile.flush()
            exit_code = process.wait()
    if exit_code != 0:
        raise RuntimeError(f"Ошибка оценки. Обучение повторять не нужно. Лог: {EVAL_LOG}")

metrics = json.loads((EVAL / "metrics.json").read_text())["datasets"]["validation"]
assert metrics["num_questions"] == 956
print(json.dumps(metrics, indent=2, ensure_ascii=False))

Итоговый архив: адаптер, отчёт, метрики, предсказания, логи, код и разбиение данных. Отправляется в тот же Comet experiment и остаётся для скачивания. Промежуточные чекпоинты находятся в Comet отдельными ZIP. Вторая переменная FULL позволяет выбрать нужный каталог вручную после перезапуска kernel.

In [ ]:
# После сброса kernel при необходимости укажите FULL вручную:
# FULL = RUN / "sft_full_YYYYMMDD_HHMMSS"
EVAL = RUN / "eval" / "validation" / FULL.name
ARCHIVE = WORK / f"{FULL.name}_artifacts.zip"
with ZipFile(ARCHIVE, "w", ZIP_DEFLATED, allowZip64=True) as z:
    z.write(FULL / "run.json", "run.json")
    for path in sorted((FULL / "adapter").rglob("*")):
        if path.is_file():
            z.write(path, "adapter/" + path.relative_to(FULL / "adapter").as_posix())
    for path in sorted(EVAL.glob("*.json*")):
        z.write(path, "evaluation/" + path.name)
    for path in sorted((RUN / "logs").glob(f"{FULL.name}*.log")):
        z.write(path, "logs/" + path.name)
    for folder in ("scripts", "src"):
        for path in sorted((REPO / folder).rglob("*.py")):
            z.write(path, "repository/" + path.relative_to(REPO).as_posix())
    z.write(REPO / "splits/v1.json", "splits/v1.json")
    freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    z.writestr("environment/pip_freeze.txt", freeze)

subprocess.run([sys.executable, "-u", "-c", r"""
import json, sys
from pathlib import Path
from comet_tracking import start_experiment
report = json.loads(Path(sys.argv[1]).read_text())
experiment = start_experiment(key=report["comet_experiment_key"])
try:
    experiment.log_asset(sys.argv[2], file_name=Path(sys.argv[2]).name, overwrite=True)
finally:
    experiment.end()
""", str(FULL / "run.json"), str(ARCHIVE)],
    env={**os.environ, "PYTHONPATH": str(REPO / "src")}, check=True)

from IPython.display import FileLink, display
print("Скачайте архив через Files:", ARCHIVE)
display(FileLink(str(ARCHIVE.relative_to(WORK))))